# <center>Code métriques 3STR</center>

Quels sont les buts de ce notebook ?  
- Mettre en forme le code sur les métriques pour le 3STR (prends en entrée une TS, une TS reconstruite et une liste d'index des dates nuageuses.  
      - Sortir des métriques occluded / observed / oa  
      - Organiser moi même comment les métriques sont faites vis-àvis des dates etcs  
- Prendre mieux en mains le code d'inférence d'U-TILISE  
      - Faire de premiers tests  
      - Voir si le système d'imputation peut-avoir un batch_size plus grand que 1   

### Inférence (reprise script eval)
Utilisation du script d'imputation pour avoir les prédictions (la ts reconstruite)

In [1]:
import collections.abc
import logging
import random
import re
from functools import partial
from typing import Any
from typing import Dict
from typing import List
from typing import Optional
from typing import Tuple
from typing import Union
from pathlib import Path
import numpy as np
import torch
import torch.utils
import torch.utils.data
from omegaconf import DictConfig
from torch import Tensor
from torch.nn import functional as F
from torch.utils.data import Dataset
from dataloader_CIRCA.datasets import CIRCA_ADAPTED2UTILISE_Dataset

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # ou FutureWarning, UserWarning, ...

In [2]:
# Définition du dataset
SUBSET_LENGTH = 20
batch_size_inference = 1

filter_settings = {
    "type": "cloud-free",  # Strategy for removing observations with data gaps.
    # ['cloud-free', 'cloud-free_consecutive']
    "min_length": 5,  # Minimum sequence length.
    "return_valid_obs_only": True,  # True to return the cloud-filtered sequences, False otherwise.
    # "max_t_sampling": 10,            # Maximum temporal sampling frequency in days.
}

mask_kwargs = {
    "mask_type": "random_clouds",  # Mask the input time series with randomly sampled cloud masks or the actual cloud masks. ['random_clouds', 'real_clouds']
    "ratio_masked_frames": 0.5,  # Ratio of partially/fully masked images per image time series (upper bound).
    "ratio_fully_masked_frames": 0.0,  # Ratio of fully masked images per image time series (upper bound).
    "fixed_masking_ratio": False,  # True to vary the masking ratio across different image time series, False otherwise.
    "non_masked_frames": [
        0
    ],  # list of int, time steps to be excluded from masking. E.g., [0] never masks the first frame in a sequence.
    "intersect_real_cloud_masks": False,  # True to intersect randomly sampled cloud masks with the actual cloud masks, False otherwise.
    "dilate_cloud_masks": False,  # True to dilate the cloud masks before masking, False otherwise.
    "fill_type": "fill_value",  # Strategy for initializing masked pixels. ['fill_value', 'white_noise', 'mean']
    "fill_value": 1,  # Pixel value of masked pixels. Used if fill_type == 'fill_value'.
    "p_filter": 0.1,
}

params_dataset = {
    'phase': "test",
    'hdf5_file': "/DATA_10TB/data_rpg/circa/hdf5/CIRCA_CR_merged.hdf5",
    'shuffle': False,
    'use_sar': 'mix_closest',
    'channels': "all",
    # U-TILISE specific parameters
    'filter_settings': filter_settings,
    'max_seq_length': 30,
    'render_occluded_above_p': None, # Set to None to keep original cloud masks. Minimum cloud cover to fully mask an input image (0.9 demo config)
    'mask_kwargs': mask_kwargs,
    'pe_strategy': "day-within-sequence",
    'augment': False,
    'process_data': True,
    'seed': 42,
    # Récupération de vieux arguments du repo
    'crop_settings': None,
    'return_cloud_mask': True,
}

dset = CIRCA_ADAPTED2UTILISE_Dataset(**params_dataset)

In [3]:
from lib.data_utils import seed_worker, pad_collate
from lib import config_utils
# dset = torch.utils.data.Subset(dset, range(SUBSET_LENGTH))
dataloader = torch.utils.data.DataLoader(
    dataset=dset,
    batch_size=batch_size_inference,
    shuffle=False,
    num_workers=0,
    collate_fn=None,
    pin_memory=False,
    drop_last=False,
)

### Chargement du modèle

In [4]:
from lib.models.utilise import UTILISE
from lib.eval_tools import impute_sequence
from lib import data_utils
temporal_window = dset.max_seq_length
num_channels = dset.num_channels
device = torch.device("cuda:0")
path_trainings_results = Path("/DATA_10TB/data_rpg/outputs/U-TILISE/results/ALL_SAR_120_epochs_2025-07-11_16-56")
path_ckpt = path_trainings_results / "checkpoints" / "Model_best.pth" 
path_config_training = path_trainings_results / "config.yaml" 
assert path_ckpt.exists()
assert path_config_training.exists()
config_training = config_utils.read_config(path_config_training)
config_training.utilise.input_dim = num_channels
config_training.utilise.output_dim = 10 # num_channels - 4 if use_sar
model = UTILISE(**config_training.utilise)
checkpoint = torch.load(path_ckpt)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device).eval()
del checkpoint

### Inférence sur un batch

In [5]:
def infer_one_batch(batch, model, temporal_window, device, t_start=None, t_end=None):
    if t_start is not None and t_end is not None:
        # Choose a subsequence
        batch["x"] = batch["x"][:, t_start:t_end, ...]
    
        for key in ["y", "masks", "cloud_mask", "masks_valid_obs"]:
            if key in batch:
                batch[key] = batch[key][:, t_start:t_end, ...]
    
        for key in ["days", "position_days"]:
            if key in batch:
                batch[key] = batch[key][:, t_start:t_end]
    
    batch = data_utils.to_device(batch, device)
    y_pred = impute_sequence(model, batch, temporal_window, return_att=False)
    batch = data_utils.to_device(batch, "cpu")
    y_pred = y_pred.cpu()
    return batch, y_pred

In [6]:
batch = next(iter(dataloader))

In [7]:
batch['y'].shape

torch.Size([1, 29, 10, 256, 256])

In [8]:
batch_processed, y_pred = infer_one_batch(
    batch=batch, 
    model=model, 
    temporal_window=temporal_window, 
    device=device, 
    t_start=0, 
    t_end=10,
)

In [9]:
batch_processed['y'].shape

torch.Size([1, 10, 10, 256, 256])

In [10]:
y_pred.shape

torch.Size([1, 10, 10, 256, 256])

### Code métriques

In [11]:
import math
from typing import Any
from typing import Dict
from typing import Literal

import torch
import torchgeometry as tgm
from prodict import Prodict
from torch import Tensor

In [12]:
class EvalMetrics:
    """
    Computes the metrics used to monitor the training progress or for evaluation.
    """
    def __init__(
        self, 
        masked_metrics: bool = False,
        sam_units: str = "rad", # "deg" or "rad"
        eval_occluded_observed: bool = True,  #  to evaluate the metrics over all pixels and separately for occluded and observed input pixels 
        mae: bool = True,
        rmse: bool = True,
        mse: bool = True,
        psnr: bool = True,
        sam: bool = True,
        ssim: bool = True,
    ):
        self.masked_metrics = masked_metrics
        # True to evaluate the metrics over all pixels and separately for occluded and observed input pixels;
        # False to evaluate the metrics over all pixels only
        self.eval_occluded_observed = eval_occluded_observed

        # MAE (mean absolute error)
        if mae:
            self.mae = lambda predicted, target: torch.mean(torch.abs(predicted - target))
        # MSE (mean squared error)
        if mse:
            self.mse = lambda predicted, target: torch.mean(torch.square(predicted - target))
        # RMSE (root mean square error)
        if rmse:
            self.rmse = lambda predicted, target: torch.sqrt(torch.mean(torch.square(predicted - target)))
        # SSIM (structural similarity index)
        if ssim:
            self.dssim = tgm.losses.SSIM(5, reduction="mean")
        # PSNR (peak signal-to-noise ratio)
        if psnr:
            self.psnr = lambda predicted, target: 20 * torch.log10(1 / self.rmse(predicted, target))
        # SAM (spectral angle mapper)
        if sam:
            self.sam_units = sam_units
            self.sam = EvalMetrics.compute_sam
    
    @staticmethod
    def compute_sam(predicted: Tensor, target: Tensor, units: Literal["deg", "rad"] = "rad") -> Tensor:
        """
        Computes the spectral angle mapper (SAM) averaged over all time steps and batch samples.
    
        Args:
            predicted:   torch.Tensor,  (n_frames x C x H x W).
            target:      torch.Tensor,  (n_frames x C x H x W).
    
        Returns:
            sam_value:   torch.Tensor, (1, ), mean spectral angle [rad].
        """
        dot_product = (predicted * target).sum(dim=1)
        predicted_norm = predicted.norm(dim=1)
        target_norm = target.norm(dim=1)
        # Compute the SAM score for all pixels with vector norm > 0
        flag = torch.logical_and(predicted_norm != 0.0, target_norm != 0.0)
        if torch.any(flag):
            spectral_angles = torch.clamp(dot_product[flag] / (predicted_norm[flag] * target_norm[flag]), -1, 1).acos()
            sam_score = torch.mean(spectral_angles)
            if units == "deg":
                sam_score *= 180 / math.pi
            return sam_score
        else:
            return None

    def __call__(self,
                 target: Tensor, 
                 masks: Tensor, 
                 predicted: Tensor) -> Dict[str, float]:
        """
        Args: 
            target:       torch.Tensor, (B x T x C x W x H); target sequence.
            masks:        torch.Tensor, (B x T x 1 x W x H); a pixel value of 0 indicates an
                          observed (non-masked) input pixel and a pixel value of 1 a masked input
                          pixel.
            cloud_mask:   torch.Tensor, (B x T x 1 x W x H), 0 indicates a non-occluded target pixel
                          and 1 an occluded target pixel.
            predicted:       torch.Tensor, (B x T x C x W x H); predicted sequence.
        """
        # Initialize metrics
        metrics = dict()

        # Concatenate batch and time dimension
        B, T, C, H, W = predicted.shape
        n_frames = B * T
        predicted = predicted.view(n_frames, C, H, W)
        target = target.view(n_frames, C, H, W)
        masks = masks.view(n_frames, 1, H, W).expand(target.shape)

        # Structural similarity index (SSIM) evaluated over all images
        if hasattr(self, "dssim"):
            dssim = self.dssim(predicted, target)  # outputs (1 - SSIM)/2; structural dissimilarity
            metrics["ssim"] = 1 - 2 * dssim

            # Structural similarity index (SSIM) evaluated over all images with data gaps
            if self.eval_occluded_observed:
                occ_images = (masks == 1.0).any(dim=-1).any(dim=-1).any(dim=-1)
                metrics["ssim_images_occluded_input_pixels"] = 1 - 2 * self.dssim(predicted[occ_images], target[occ_images])
                metrics["ssim_images_observed_input_pixels"] = 1 - 2 * self.dssim(predicted[~occ_images], target[~occ_images])

        # if self.masked_metrics == False: metrics are computed over all output pixels
        # if self.masked_metrics == True: metrics are computed over all non-occluded target pixels (according to GT cloud masks)
        # if self.masked_metrics:
        #     cloud_mask = cloud_mask.view(n_frames, 1, H, W)

        #     # Evaluate non-occluded target pixels only
        #     flag = cloud_mask.permute(0, 2, 3, 1).reshape(n_frames * H * W) == 0.0
        #     # print(flag.shape)
        #     # Tensor shapes: (n_frames * H * W, C)
        #     predicted = predicted.permute(0, 2, 3, 1).reshape(n_frames * H * W, C)[flag]
        #     target = target.permute(0, 2, 3, 1).reshape(n_frames * H * W, C)[flag]
        #     masks = masks.permute(0, 2, 3, 1).reshape(n_frames * H * W, C)[flag]

        # MAE (mean absolute error) evaluated over all pixels in the input sequence
        if hasattr(self, "mae"):
            metrics[f"mae"] = self.mae(predicted, target)
            if self.eval_occluded_observed:
                metrics[f"mae_occluded_input_pixels"] = self.mae(predicted[masks == 1.0], target[masks == 1.0])
                metrics[f"mae_observed_input_pixels"] = self.mae(predicted[masks == 0.0], target[masks == 0.0])

        # Root mean squared error (RMSE)
        if hasattr(self, "rmse"):
            metrics[f"rmse"] = self.rmse(predicted, target)
            if self.eval_occluded_observed:
                metrics[f"rmse_occluded_input_pixels"] = self.rmse(predicted[masks == 1.0], target[masks == 1.0])
                metrics[f"rmse_observed_input_pixels"] = self.rmse(predicted[masks == 0.0], target[masks == 0.0])

        # Mean squared error (MSE)
        if hasattr(self, "mse"):
            metrics[f"mse"] = self.mse(predicted, target)
            if self.eval_occluded_observed:
                metrics[f"mse_occluded_input_pixels"] = self.mse(predicted[masks == 1.0], target[masks == 1.0])
                metrics[f"mse_observed_input_pixels"] = self.mse(predicted[masks == 0.0], target[masks == 0.0])

        # PSNR
        if hasattr(self, "psnr"):
            metrics[f"psnr"] = self.psnr(predicted, target)
            if self.eval_occluded_observed:
                metrics[f"psnr_occluded_input_pixels"] = self.psnr(predicted[masks == 1.0], target[masks == 1.0])
                metrics[f"psnr_observed_input_pixels"] = self.psnr(predicted[masks == 0.0], target[masks == 0.0])

        # SAM
        if hasattr(self, "sam"):
            metrics[f"sam"] = self.compute_sam(predicted, target, units=self.sam_units)
            if self.eval_occluded_observed:
                metrics[f"sam_occluded_input_pixels"] = self.sam(predicted[:, :, (masks == 1.0).all(dim=1), :], target[:, :, (masks == 1.0).all(dim=1), :], units=self.sam_units)
                metrics[f"sam_observed_input_pixels"] = self.sam(predicted[:, :, (masks == 0.0).all(dim=1), :], target[:, :, (masks == 0.0).all(dim=1), :], units=self.sam_units)

        for key, value in metrics.items():
            metrics[key] = value.item()

        return metrics

In [13]:
compute_metrics = EvalMetrics(sam=False)

metrics_on_sample = compute_metrics(
    target=batch["y"], 
    masks=batch["masks"], 
    predicted=y_pred)

print("**Metrics on the sample:**")
for k, v in metrics_on_sample.items():
    print(f"{k}: {v:.4f}")

**Metrics on the sample:**
ssim: 0.9078
ssim_images_occluded_input_pixels: 0.8397
ssim_images_observed_input_pixels: 0.9759
mae: 0.0604
mae_occluded_input_pixels: 0.1311
mae_observed_input_pixels: 0.0106
rmse: 0.1323
rmse_occluded_input_pixels: 0.2049
rmse_observed_input_pixels: 0.0160
mse: 0.0175
mse_occluded_input_pixels: 0.0420
mse_observed_input_pixels: 0.0003
psnr: 17.5675
psnr_occluded_input_pixels: 13.7703
psnr_observed_input_pixels: 35.8972


In [90]:
masks=batch["masks"]
target=batch["y"]
predicted=y_pred

print(masks.shape)
# print(predicted.shape)

B, T, C, H, W = predicted.shape
n_frames = B * T
predicted = predicted.view(n_frames, C, H, W)
# target = target.view(n_frames, C, H, W)
masks = masks.view(n_frames, 1, H, W)
        

print(masks.shape)
# masks = masks.expand(target.shape)

print(masks.shape)
# print(predicted.shape)

torch.Size([1, 10, 1, 256, 256])
torch.Size([10, 1, 256, 256])
torch.Size([10, 1, 256, 256])


In [1]:
import numpy as np
np.iinfo(np.uint8).max

255

# Rapport de Métriques — Cloud Reconstruction U-TILISE

**Source des résultats :** logs SLURM dans `/mnt/stores/store_dai/tmp/speillet/logs`

## 1. Récapitulatif Global

### Masquage : Random Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | 246.3 | 513.4 | 30.48 | 0.7515 | 0.0664 | 0.7896 |
| asc+desc, random_clouds | 239.6 | 512.5 | 30.83 | 0.7711 | 0.0695 | 0.7869 |
| asc+desc, random_fully_masked, DA | 332.9 | 534.7 | 31.30 | 0.7202 | 0.0404 | 0.8343 |
| mix_closest, random_fully_masked | 375.2 | 585.6 | 31.01 | 0.7167 | 0.0416 | 0.8306 |
| mix_closest, random_fully_masked, DA | 389.5 | 610.2 | 31.21 | 0.7357 | 0.0382 | 0.8353 |
| coherence_only | 243.9 | 504.7 | 30.23 | 0.7048 | 0.0696 | 0.7935 |

### Masquage : Consecutive Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | 364.2 | 819.4 | 26.51 | 0.6881 | 0.0987 | 0.6286 |
| asc+desc, random_clouds | 355.0 | 810.5 | 26.95 | 0.7086 | 0.1284 | 0.6326 |
| asc+desc, random_fully_masked, DA | 339.4 | 576.2 | 30.56 | 0.6832 | 0.0395 | 0.8166 |
| mix_closest, random_fully_masked | 388.6 | 640.2 | 29.93 | 0.6772 | 0.0406 | 0.8054 |
| mix_closest, random_fully_masked, DA | — | — | — | — | — | — |
| coherence_only | 371.0 | 826.0 | 26.16 | 0.6488 | 0.1002 | 0.6264 |

## 2. Occluded vs Observed

### Masquage : Random Fully Masked

| Modèle | Type | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | Occluded | 998.2 | 1336.7 | 23.21 | — | 0.2314 | 0.3926 |
| asc+desc, random_clouds, DA | Observed | 64.3 | 93.8 | 40.71 | — | 0.0271 | 0.9906 |
| asc+desc, random_clouds | Occluded | 983.2 | 1336.8 | 23.72 | — | 0.2681 | 0.3750 |
| asc+desc, random_clouds | Observed | 56.1 | 82.1 | 41.83 | — | 0.0234 | 0.9932 |
| asc+desc, random_fully_masked, DA | Occluded | 1055.5 | 1325.0 | 21.31 | — | 0.1014 | 0.6231 |
| asc+desc, random_fully_masked, DA | Observed | 67.7 | 98.3 | 40.33 | — | 0.0290 | 0.9891 |
| mix_closest, random_fully_masked | Occluded | 1140.2 | 1412.7 | 21.09 | — | 0.1036 | 0.6235 |
| mix_closest, random_fully_masked | Observed | 71.1 | 102.9 | 39.94 | — | 0.0300 | 0.9877 |
| mix_closest, random_fully_masked, DA | Occluded | 1151.2 | 1440.3 | 21.14 | — | 0.1010 | 0.6283 |
| mix_closest, random_fully_masked, DA | Observed | 64.7 | 94.5 | 40.64 | — | 0.0263 | 0.9912 |
| coherence_only | Occluded | 984.0 | 1314.7 | 22.04 | — | 0.2360 | 0.4000 |
| coherence_only | Observed | 71.3 | 103.8 | 39.83 | — | 0.0299 | 0.9883 |

### Masquage : Consecutive Fully Masked

| Modèle | Type | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | Occluded | 2296.9 | 2595.1 | 16.45 | — | 0.6064 | 0.0721 |
| asc+desc, random_clouds, DA | Observed | 64.5 | 94.2 | 40.67 | — | 0.0271 | 0.9905 |
| asc+desc, random_clouds | Occluded | 2266.2 | 2570.1 | 17.09 | — | 0.8594 | 0.0573 |
| asc+desc, random_clouds | Observed | 56.3 | 82.3 | 41.81 | — | 0.0234 | 0.9932 |
| asc+desc, random_fully_masked, DA | Occluded | 1157.0 | 1512.9 | 19.39 | — | 0.1122 | 0.5013 |
| asc+desc, random_fully_masked, DA | Observed | 67.9 | 98.5 | 40.31 | — | 0.0290 | 0.9890 |
| mix_closest, random_fully_masked | Occluded | 1305.4 | 1650.9 | 18.62 | — | 0.1147 | 0.4997 |
| mix_closest, random_fully_masked | Observed | 71.3 | 103.1 | 39.92 | — | 0.0300 | 0.9877 |
| mix_closest, random_fully_masked, DA | Occluded | — | — | — | — | — | — |
| mix_closest, random_fully_masked, DA | Observed | — | — | — | — | — | — |
| coherence_only | Occluded | 2309.3 | 2608.6 | 15.12 | — | 0.5966 | 0.1403 |
| coherence_only | Observed | 71.6 | 104.2 | 39.79 | — | 0.0300 | 0.9883 |

## 3. Métriques par bande spectrale

### SSIM par bande

#### Masquage : Random Fully Masked

| Modèle | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | 0.6030 | 0.6742 | 0.6824 | 0.7641 | 0.8065 | 0.8238 | 0.7683 | 0.8239 | 0.7958 | 0.7724 |
| asc+desc, random_clouds | 0.6465 | 0.7092 | 0.7203 | 0.7751 | 0.8118 | 0.8299 | 0.8036 | 0.8282 | 0.8044 | 0.7825 |
| asc+desc, random_fully_masked, DA | 0.5688 | 0.6508 | 0.6544 | 0.7270 | 0.7681 | 0.7863 | 0.7510 | 0.7888 | 0.7631 | 0.7442 |
| mix_closest, random_fully_masked | 0.5599 | 0.6461 | 0.6525 | 0.7181 | 0.7644 | 0.7838 | 0.7468 | 0.7834 | 0.7687 | 0.7437 |
| mix_closest, random_fully_masked, DA | 0.5957 | 0.6732 | 0.6804 | 0.7427 | 0.7731 | 0.7946 | 0.7687 | 0.7915 | 0.7780 | 0.7596 |
| coherence_only | 0.5442 | 0.6173 | 0.6230 | 0.7065 | 0.7719 | 0.7916 | 0.7376 | 0.7951 | 0.7420 | 0.7191 |

#### Masquage : Consecutive Fully Masked

| Modèle | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | 0.5554 | 0.6186 | 0.6281 | 0.6980 | 0.7363 | 0.7528 | 0.7033 | 0.7521 | 0.7281 | 0.7087 |
| asc+desc, random_clouds | 0.5994 | 0.6545 | 0.6665 | 0.7108 | 0.7423 | 0.7595 | 0.7379 | 0.7569 | 0.7384 | 0.7200 |
| asc+desc, random_fully_masked, DA | 0.5407 | 0.6169 | 0.6228 | 0.6859 | 0.7279 | 0.7461 | 0.7141 | 0.7472 | 0.7246 | 0.7061 |
| mix_closest, random_fully_masked | 0.5298 | 0.6106 | 0.6189 | 0.6737 | 0.7221 | 0.7415 | 0.7086 | 0.7403 | 0.7242 | 0.7021 |
| mix_closest, random_fully_masked, DA | — | — | — | — | — | — | — | — | — | — |
| coherence_only | 0.5041 | 0.5698 | 0.5769 | 0.6494 | 0.7073 | 0.7260 | 0.6789 | 0.7282 | 0.6827 | 0.6643 |

### PSNR par bande

#### Masquage : Random Fully Masked

| Modèle | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | 34.46 | 34.39 | 32.76 | 32.99 | 30.22 | 29.85 | 27.67 | 29.22 | 30.95 | 31.12 |
| asc+desc, random_clouds | 35.51 | 34.84 | 33.39 | 33.45 | 30.30 | 29.91 | 28.00 | 29.27 | 31.08 | 31.46 |
| asc+desc, random_fully_masked, DA | 34.53 | 35.53 | 33.33 | 33.43 | 30.97 | 30.64 | 28.76 | 30.10 | 32.23 | 32.08 |
| mix_closest, random_fully_masked | 34.17 | 35.18 | 32.95 | 33.01 | 30.67 | 30.29 | 28.58 | 29.82 | 31.83 | 31.84 |
| mix_closest, random_fully_masked, DA | 34.67 | 35.39 | 33.37 | 34.09 | 30.72 | 30.25 | 28.56 | 29.68 | 31.72 | 32.40 |
| coherence_only | 33.87 | 34.31 | 32.47 | 32.24 | 30.00 | 29.69 | 27.70 | 29.14 | 30.42 | 30.79 |

#### Masquage : Consecutive Fully Masked

| Modèle | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | 29.70 | 29.59 | 28.78 | 28.41 | 26.04 | 25.89 | 23.90 | 25.19 | 26.85 | 27.59 |
| asc+desc, random_clouds | 30.62 | 30.08 | 29.43 | 28.97 | 26.28 | 26.11 | 24.35 | 25.41 | 27.10 | 27.96 |
| asc+desc, random_fully_masked, DA | 32.45 | 33.60 | 32.13 | 32.01 | 30.26 | 30.19 | 28.35 | 29.60 | 31.70 | 31.76 |
| mix_closest, random_fully_masked | 31.80 | 32.84 | 31.48 | 31.17 | 29.47 | 29.43 | 27.84 | 28.99 | 30.99 | 31.23 |
| mix_closest, random_fully_masked, DA | — | — | — | — | — | — | — | — | — | — |
| coherence_only | 29.13 | 29.34 | 28.41 | 27.64 | 25.67 | 25.59 | 23.80 | 24.99 | 26.29 | 27.22 |

### MAE par bande

#### Masquage : Random Fully Masked

| Modèle | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | 159.2 | 165.7 | 184.5 | 200.6 | 283.0 | 304.8 | 352.0 | 326.4 | 261.0 | 225.7 |
| asc+desc, random_clouds | 142.9 | 160.0 | 174.2 | 193.5 | 280.2 | 301.5 | 341.7 | 323.5 | 259.7 | 219.1 |
| asc+desc, random_fully_masked, DA | 228.3 | 227.0 | 251.6 | 281.5 | 386.4 | 413.8 | 461.8 | 441.1 | 340.0 | 297.0 |
| mix_closest, random_fully_masked | 265.6 | 266.0 | 292.8 | 327.7 | 432.3 | 461.4 | 503.6 | 482.5 | 384.3 | 335.6 |
| mix_closest, random_fully_masked, DA | 283.4 | 287.3 | 315.3 | 336.5 | 425.3 | 459.0 | 516.9 | 495.4 | 427.8 | 347.8 |
| coherence_only | 165.6 | 161.0 | 184.1 | 204.9 | 278.1 | 296.9 | 342.3 | 317.5 | 262.9 | 225.7 |

#### Masquage : Consecutive Fully Masked

| Modèle | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | 242.5 | 257.6 | 269.6 | 306.9 | 419.6 | 449.9 | 502.3 | 481.8 | 389.9 | 321.5 |
| asc+desc, random_clouds | 224.1 | 249.8 | 257.0 | 297.0 | 415.3 | 444.6 | 488.7 | 475.6 | 384.1 | 313.9 |
| asc+desc, random_fully_masked, DA | 243.4 | 239.8 | 260.8 | 293.1 | 392.3 | 414.7 | 463.1 | 443.0 | 344.6 | 299.5 |
| mix_closest, random_fully_masked | 285.6 | 285.4 | 307.4 | 347.0 | 448.4 | 471.3 | 512.8 | 491.5 | 393.1 | 343.5 |
| mix_closest, random_fully_masked, DA | — | — | — | — | — | — | — | — | — | — |
| coherence_only | 255.1 | 259.8 | 275.2 | 319.7 | 427.5 | 455.3 | 504.8 | 485.5 | 399.1 | 328.0 |

## 4. Disponibilité des résultats

| Modèle | Random Fully Masked | Consecutive Fully Masked |
|:---|:---:|:---:|
| asc+desc, random_clouds, DA | ✅ | ✅ |
| asc+desc, random_clouds | ✅ | ✅ |
| asc+desc, random_fully_masked, DA | ✅ | ✅ |
| mix_closest, random_fully_masked | ✅ | ✅ |
| mix_closest, random_fully_masked, DA | ✅ | ❌ |
| coherence_only | ✅ | ✅ |


# Rapport de Métriques — Cloud Reconstruction U-TILISE

**Source des résultats :** logs SLURM dans `/mnt/stores/store_dai/tmp/speillet/logs`

## 1. Récapitulatif Global

### Masquage : Random Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | 246.3 | 513.4 | 30.48 | 0.7515 | 0.0664 | 0.7896 |
| asc+desc, random_clouds | 239.6 | 512.5 | 30.83 | 0.7711 | 0.0695 | 0.7869 |
| asc+desc, random_fully_masked, DA | 332.9 | 534.7 | 31.30 | 0.7202 | 0.0404 | 0.8343 |
| mix_closest, random_fully_masked | 375.2 | 585.6 | 31.01 | 0.7167 | 0.0416 | 0.8306 |
| mix_closest, random_fully_masked, DA | 389.5 | 610.2 | 31.21 | 0.7357 | 0.0382 | 0.8353 |
| coherence_only | 243.9 | 504.7 | 30.23 | 0.7048 | 0.0696 | 0.7935 |

### Masquage : Consecutive Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | 364.2 | 819.4 | 26.51 | 0.6881 | 0.0987 | 0.6286 |
| asc+desc, random_clouds | 355.0 | 810.5 | 26.95 | 0.7086 | 0.1284 | 0.6326 |
| asc+desc, random_fully_masked, DA | 339.4 | 576.2 | 30.56 | 0.6832 | 0.0395 | 0.8166 |
| mix_closest, random_fully_masked | 388.6 | 640.2 | 29.93 | 0.6772 | 0.0406 | 0.8054 |
| mix_closest, random_fully_masked, DA | 421.1 | 709.6 | 29.47 | 0.6861 | 0.0399 | 0.7864 |
| coherence_only | 371.0 | 826.0 | 26.16 | 0.6488 | 0.1002 | 0.6264 |

## 2. Métriques sur pixels reconstruits (Occluded)

### Masquage : Random Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | 998.2 | 1336.7 | 23.21 | 0.4328 | 0.2314 | 0.3926 |
| asc+desc, random_clouds | 983.2 | 1336.8 | 23.72 | 0.4275 | 0.2681 | 0.3750 |
| asc+desc, random_fully_masked, DA | 1055.5 | 1325.0 | 21.31 | 0.4019 | 0.1014 | 0.6231 |
| mix_closest, random_fully_masked | 1140.2 | 1412.7 | 21.09 | 0.4173 | 0.1036 | 0.6235 |
| mix_closest, random_fully_masked, DA | 1151.2 | 1440.3 | 21.14 | 0.4576 | 0.1010 | 0.6283 |
| coherence_only | 984.0 | 1314.7 | 22.04 | 0.3466 | 0.2360 | 0.4000 |

### Masquage : Consecutive Fully Masked

| Modèle | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | 2296.9 | 2595.1 | 16.45 | 0.1560 | 0.6064 | 0.0721 |
| asc+desc, random_clouds | 2266.2 | 2570.1 | 17.09 | 0.1581 | 0.8594 | 0.0573 |
| asc+desc, random_fully_masked, DA | 1157.0 | 1512.9 | 19.39 | 0.2904 | 0.1122 | 0.5013 |
| mix_closest, random_fully_masked | 1305.4 | 1650.9 | 18.62 | 0.2937 | 0.1147 | 0.4997 |
| mix_closest, random_fully_masked, DA | 1437.6 | 1822.1 | 17.70 | 0.2879 | 0.1353 | 0.4253 |
| coherence_only | 2309.3 | 2608.6 | 15.12 | 0.1041 | 0.5966 | 0.1403 |

## 3. Occluded vs Observed (détail)

### Masquage : Random Fully Masked

| Modèle | Type | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | Occluded | 998.2 | 1336.7 | 23.21 | 0.4328 | 0.2314 | 0.3926 |
| asc+desc, random_clouds, DA | Observed | 64.3 | 93.8 | 40.71 | 0.8186 | 0.0271 | 0.9906 |
| asc+desc, random_clouds | Occluded | 983.2 | 1336.8 | 23.72 | 0.4275 | 0.2681 | 0.3750 |
| asc+desc, random_clouds | Observed | 56.1 | 82.1 | 41.83 | 0.8450 | 0.0234 | 0.9932 |
| asc+desc, random_fully_masked, DA | Occluded | 1055.5 | 1325.0 | 21.31 | 0.4019 | 0.1014 | 0.6231 |
| asc+desc, random_fully_masked, DA | Observed | 67.7 | 98.3 | 40.33 | 0.8276 | 0.0290 | 0.9891 |
| mix_closest, random_fully_masked | Occluded | 1140.2 | 1412.7 | 21.09 | 0.4173 | 0.1036 | 0.6235 |
| mix_closest, random_fully_masked | Observed | 71.1 | 102.9 | 39.94 | 0.8191 | 0.0300 | 0.9877 |
| mix_closest, random_fully_masked, DA | Occluded | 1151.2 | 1440.3 | 21.14 | 0.4576 | 0.1010 | 0.6283 |
| mix_closest, random_fully_masked, DA | Observed | 64.7 | 94.5 | 40.64 | 0.8339 | 0.0263 | 0.9912 |
| coherence_only | Occluded | 984.0 | 1314.7 | 22.04 | 0.3466 | 0.2360 | 0.4000 |
| coherence_only | Observed | 71.3 | 103.8 | 39.83 | 0.8044 | 0.0299 | 0.9883 |

### Masquage : Consecutive Fully Masked

| Modèle | Type | MAE | RMSE | PSNR | SSIM | SAM | R2 |
|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | Occluded | 2296.9 | 2595.1 | 16.45 | 0.1560 | 0.6064 | 0.0721 |
| asc+desc, random_clouds, DA | Observed | 64.5 | 94.2 | 40.67 | 0.8180 | 0.0271 | 0.9905 |
| asc+desc, random_clouds | Occluded | 2266.2 | 2570.1 | 17.09 | 0.1581 | 0.8594 | 0.0573 |
| asc+desc, random_clouds | Observed | 56.3 | 82.3 | 41.81 | 0.8445 | 0.0234 | 0.9932 |
| asc+desc, random_fully_masked, DA | Occluded | 1157.0 | 1512.9 | 19.39 | 0.2904 | 0.1122 | 0.5013 |
| asc+desc, random_fully_masked, DA | Observed | 67.9 | 98.5 | 40.31 | 0.8272 | 0.0290 | 0.9890 |
| mix_closest, random_fully_masked | Occluded | 1305.4 | 1650.9 | 18.62 | 0.2937 | 0.1147 | 0.4997 |
| mix_closest, random_fully_masked | Observed | 71.3 | 103.1 | 39.92 | 0.8185 | 0.0300 | 0.9877 |
| mix_closest, random_fully_masked, DA | Occluded | 1437.6 | 1822.1 | 17.70 | 0.2879 | 0.1353 | 0.4253 |
| mix_closest, random_fully_masked, DA | Observed | 65.1 | 94.9 | 40.60 | 0.8332 | 0.0264 | 0.9912 |
| coherence_only | Occluded | 2309.3 | 2608.6 | 15.12 | 0.1041 | 0.5966 | 0.1403 |
| coherence_only | Observed | 71.6 | 104.2 | 39.79 | 0.8038 | 0.0300 | 0.9883 |

## 4. Métriques par bande spectrale

### SSIM par bande

#### Masquage : Random Fully Masked

| Modèle | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | 0.6030 | 0.6742 | 0.6824 | 0.7641 | 0.8065 | 0.8238 | 0.7683 | 0.8239 | 0.7958 | 0.7724 |
| asc+desc, random_clouds | 0.6465 | 0.7092 | 0.7203 | 0.7751 | 0.8118 | 0.8299 | 0.8036 | 0.8282 | 0.8044 | 0.7825 |
| asc+desc, random_fully_masked, DA | 0.5688 | 0.6508 | 0.6544 | 0.7270 | 0.7681 | 0.7863 | 0.7510 | 0.7888 | 0.7631 | 0.7442 |
| mix_closest, random_fully_masked | 0.5599 | 0.6461 | 0.6525 | 0.7181 | 0.7644 | 0.7838 | 0.7468 | 0.7834 | 0.7687 | 0.7437 |
| mix_closest, random_fully_masked, DA | 0.5957 | 0.6732 | 0.6804 | 0.7427 | 0.7731 | 0.7946 | 0.7687 | 0.7915 | 0.7780 | 0.7596 |
| coherence_only | 0.5442 | 0.6173 | 0.6230 | 0.7065 | 0.7719 | 0.7916 | 0.7376 | 0.7951 | 0.7420 | 0.7191 |

#### Masquage : Consecutive Fully Masked

| Modèle | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | 0.5554 | 0.6186 | 0.6281 | 0.6980 | 0.7363 | 0.7528 | 0.7033 | 0.7521 | 0.7281 | 0.7087 |
| asc+desc, random_clouds | 0.5994 | 0.6545 | 0.6665 | 0.7108 | 0.7423 | 0.7595 | 0.7379 | 0.7569 | 0.7384 | 0.7200 |
| asc+desc, random_fully_masked, DA | 0.5407 | 0.6169 | 0.6228 | 0.6859 | 0.7279 | 0.7461 | 0.7141 | 0.7472 | 0.7246 | 0.7061 |
| mix_closest, random_fully_masked | 0.5298 | 0.6106 | 0.6189 | 0.6737 | 0.7221 | 0.7415 | 0.7086 | 0.7403 | 0.7242 | 0.7021 |
| mix_closest, random_fully_masked, DA | 0.5520 | 0.6249 | 0.6344 | 0.6876 | 0.7230 | 0.7455 | 0.7217 | 0.7414 | 0.7230 | 0.7078 |
| coherence_only | 0.5041 | 0.5698 | 0.5769 | 0.6494 | 0.7073 | 0.7260 | 0.6789 | 0.7282 | 0.6827 | 0.6643 |

### PSNR par bande

#### Masquage : Random Fully Masked

| Modèle | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | 34.46 | 34.39 | 32.76 | 32.99 | 30.22 | 29.85 | 27.67 | 29.22 | 30.95 | 31.12 |
| asc+desc, random_clouds | 35.51 | 34.84 | 33.39 | 33.45 | 30.30 | 29.91 | 28.00 | 29.27 | 31.08 | 31.46 |
| asc+desc, random_fully_masked, DA | 34.53 | 35.53 | 33.33 | 33.43 | 30.97 | 30.64 | 28.76 | 30.10 | 32.23 | 32.08 |
| mix_closest, random_fully_masked | 34.17 | 35.18 | 32.95 | 33.01 | 30.67 | 30.29 | 28.58 | 29.82 | 31.83 | 31.84 |
| mix_closest, random_fully_masked, DA | 34.67 | 35.39 | 33.37 | 34.09 | 30.72 | 30.25 | 28.56 | 29.68 | 31.72 | 32.40 |
| coherence_only | 33.87 | 34.31 | 32.47 | 32.24 | 30.00 | 29.69 | 27.70 | 29.14 | 30.42 | 30.79 |

#### Masquage : Consecutive Fully Masked

| Modèle | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | 29.70 | 29.59 | 28.78 | 28.41 | 26.04 | 25.89 | 23.90 | 25.19 | 26.85 | 27.59 |
| asc+desc, random_clouds | 30.62 | 30.08 | 29.43 | 28.97 | 26.28 | 26.11 | 24.35 | 25.41 | 27.10 | 27.96 |
| asc+desc, random_fully_masked, DA | 32.45 | 33.60 | 32.13 | 32.01 | 30.26 | 30.19 | 28.35 | 29.60 | 31.70 | 31.76 |
| mix_closest, random_fully_masked | 31.80 | 32.84 | 31.48 | 31.17 | 29.47 | 29.43 | 27.84 | 28.99 | 30.99 | 31.23 |
| mix_closest, random_fully_masked, DA | 31.45 | 32.21 | 30.95 | 31.39 | 29.19 | 29.03 | 27.44 | 28.44 | 29.57 | 30.87 |
| coherence_only | 29.13 | 29.34 | 28.41 | 27.64 | 25.67 | 25.59 | 23.80 | 24.99 | 26.29 | 27.22 |

### MAE par bande

#### Masquage : Random Fully Masked

| Modèle | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | 159.2 | 165.7 | 184.5 | 200.6 | 283.0 | 304.8 | 352.0 | 326.4 | 261.0 | 225.7 |
| asc+desc, random_clouds | 142.9 | 160.0 | 174.2 | 193.5 | 280.2 | 301.5 | 341.7 | 323.5 | 259.7 | 219.1 |
| asc+desc, random_fully_masked, DA | 228.3 | 227.0 | 251.6 | 281.5 | 386.4 | 413.8 | 461.8 | 441.1 | 340.0 | 297.0 |
| mix_closest, random_fully_masked | 265.6 | 266.0 | 292.8 | 327.7 | 432.3 | 461.4 | 503.6 | 482.5 | 384.3 | 335.6 |
| mix_closest, random_fully_masked, DA | 283.4 | 287.3 | 315.3 | 336.5 | 425.3 | 459.0 | 516.9 | 495.4 | 427.8 | 347.8 |
| coherence_only | 165.6 | 161.0 | 184.1 | 204.9 | 278.1 | 296.9 | 342.3 | 317.5 | 262.9 | 225.7 |

#### Masquage : Consecutive Fully Masked

| Modèle | B2 | B3 | B4 | B5 | B6 | B7 | B8 | B8A | B11 | B12 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| asc+desc, random_clouds, DA | 242.5 | 257.6 | 269.6 | 306.9 | 419.6 | 449.9 | 502.3 | 481.8 | 389.9 | 321.5 |
| asc+desc, random_clouds | 224.1 | 249.8 | 257.0 | 297.0 | 415.3 | 444.6 | 488.7 | 475.6 | 384.1 | 313.9 |
| asc+desc, random_fully_masked, DA | 243.4 | 239.8 | 260.8 | 293.1 | 392.3 | 414.7 | 463.1 | 443.0 | 344.6 | 299.5 |
| mix_closest, random_fully_masked | 285.6 | 285.4 | 307.4 | 347.0 | 448.4 | 471.3 | 512.8 | 491.5 | 393.1 | 343.5 |
| mix_closest, random_fully_masked, DA | 321.4 | 324.7 | 351.8 | 373.8 | 451.2 | 480.1 | 539.6 | 518.3 | 474.1 | 376.3 |
| coherence_only | 255.1 | 259.8 | 275.2 | 319.7 | 427.5 | 455.3 | 504.8 | 485.5 | 399.1 | 328.0 |

## 5. Disponibilité des résultats

| Modèle | Random Fully Masked | Consecutive Fully Masked |
|:---|:---:|:---:|
| asc+desc, random_clouds, DA | ✅ | ✅ |
| asc+desc, random_clouds | ✅ | ✅ |
| asc+desc, random_fully_masked, DA | ✅ | ✅ |
| mix_closest, random_fully_masked | ✅ | ✅ |
| mix_closest, random_fully_masked, DA | ✅ | ✅ |
| coherence_only | ✅ | ✅ |
